# CAMELS: Time Series for the Website
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 05-05-2026<br>

**Introduction:**<br>
This script combines the daily discharge records with the daily basin meteorology computed from the EMO-1 dataset and expormeteo a time series per gauging station to be plotted in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
from ocab.plots.stations import plot_station_timeseries, create_station_html


## Configuration


In [2]:
cfg = Config('config_CAMELS_v200.yml')

# use this meteo dataset
meteo_ds = 'ROCIO-IBEB' # EMO1

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_web = Path('../../docs')
path_layers = path_web / 'layers'
path_ts = path_web / 'timeseries' / 'stations'
path_plots = path_ts / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# point layer
filename = 'stations.geojson'

# decimals in output timeseries
rounding = {
    'discharge_cms': 3,
    'discharge_mm': 1,
    'temp_degC': 1,
    'precip_mm': 1,
    'pet_mm': 1,
}

variables = {
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}



## Create time series


In [3]:
# load points
points = gpd.read_file(cfg.path_gis / filename).set_index('id')

# process timeseries for each station
for ID in tqdm(points.index, desc='points'):
        
    # discharge timeseries
    try:
        dis = pd.read_parquet(path_in / 'discharge' / f'{ID:04d}.parquet')
        dis.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        dis['discharge_mm'] = dis['discharge_cms'] / points.loc[ID, 'catch_skm'] * 86400 / 1000
        # add annual discharge to the attributes
        points.loc[ID, 'discharge_mm'] = dis['discharge_mm'].mean(skipna=True) * 365
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue

    # meteo timeseries
    try:
        try:
            meteo = pd.read_parquet(path_in / 'meteo' / meteo_ds / f'{ID:04d}.parquet').loc[ID]
        except:
            meteo = pd.read_parquet(path_in / 'meteo' / 'EMO1' / f'{ID:04d}.parquet').loc[ID]
            meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.rename(columns=variables, inplace=True, errors='ignore')
        # correct dates
        if meteo_ds == 'EMO1':
            meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
    except Exception as e:
        logger.error(f'Loading meteo timeseries for station {ID:04d}: {e}')
        continue

    # merge timeseries
    start = max(cfg.start, meteo.first_valid_index(), dis.first_valid_index())
    end = min(cfg.end, meteo.last_valid_index(), dis.last_valid_index())
    ts = pd.concat([dis.loc[start:end], meteo.loc[start:end]], axis=1)
    ts = ts[ts.columns.intersection(rounding)].round(rounding)

    # export timeseries
    ts.to_parquet(path_ts / f'{ID:04d}.parquet')

    # extract attributes
    attrs = points.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title(), 
            attrs['river'].title(), 
            attrs['basin'].title()
        )
        
        fig = plot_station_timeseries(
            ts,
            area=attrs['catch_skm'],
            title=title,
            regime=attrs['regime'],
            save=True
        )
        
        # save plot as HTML
        create_station_html(
            fig, 
            path=path_plots / f'{ID}.html', 
            start=ts.index.min().strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except:
        print(f"The plot for time series {ID} couldn't be created")

# export updated point layer
points['discharge_mm'] = points['discharge_mm'].round(0).astype(int)
points.to_file(path_layers / filename)

points:   0%|          | 0/1101 [00:00<?, ?it/s]

The plot for time series 6058 couldn't be created
The plot for time series 6224 couldn't be created
